### cleaning and normalization

### 1) Dermstore
1) Fix brand casing so "Neostrata" and "NEOSTRATA" aren't treated as two brands
2) Convert price from text ("188.0") to a real number
3) Trim extra spaces from text fields like description
4) Split the ingredient lists properly (they mix commas and line breaks)
5) Keep the messy concern labels for now, just flagged — we handle those in the transformation step
6) Drop the 6 products that have no ingredients (can't recommend those)

In [ ]:
df_derm = spark.read.json("/Volumes/dermstore/dermstoreschema/dermstorevolume/myfiles/dermstore/dermstore_data.json")
display (df_derm)

In [0]:
df_derm.select("brand").distinct().display()

brand
By Terry
NUDESTIX
jane iredale
Replenix
Paula's Choice
EVOLVh
Bioderma
Kevyn Aucoin
Chantecaille
Erborian


In [0]:

# 1) — Fix brand casing

from pyspark.sql import functions as F

# First, SEE the casing problem: count distinct brands as-is,
# then compare to distinct brands after normalizing case.
print("Distinct brands BEFORE:", df_derm.select("brand").distinct().count())

# initcap() converts "NEOSTRATA" and "neostrata" both -> "Neostrata"
# trim() removes any stray leading/trailing spaces first.
df_derm = df_derm.withColumn("brand", F.initcap(F.trim(F.col("brand"))))

print("Distinct brands AFTER: ", df_derm.select("brand").distinct().count())

# Show the cleaned distinct brand list
df_derm.select("brand").distinct().orderBy("brand").show(100, truncate=False)

Distinct brands BEFORE: 86
Distinct brands AFTER:  86
+--------------------------+
|brand                     |
+--------------------------+
|Agent Nateur              |
|Alterna                   |
|Anastasia Beverly Hills   |
|Arcona                    |
|Augustinus Bader          |
|Bioderma                  |
|Briogeo                   |
|By Terry                  |
|Caswell-massey            |
|Chantecaille              |
|Christophe Robin          |
|Clarityrx                 |
|Colorescience             |
|Coola                     |
|Dermaflash                |
|Dermalogica               |
|Dermaquest                |
|Dhc                       |
|Doctor Rogers             |
|Dphue                     |
|Elizabeth Arden           |
|Eltamd                    |
|Eminence Organic Skin Care|
|Epionce                   |
|Erborian                  |
|Espa                      |
|Estée Lauder              |
|Evolvh                    |
|Filorga                   |
|Ghd              

### Convert price from text ("188.0") to a real number


In [0]:
# ============================================================
# Dermstore cleaning — Step 2: Convert price from text to number
# ============================================================
# Why: price comes in as text ("188.0"), so the computer can't do
# math with it (no sorting, no "under $50" filtering). We convert
# it to a real number so price becomes usable.

# 1) Look at the prices before converting (all clean plain numbers)
df_derm.select("price").distinct().display()

# 2) Convert the price column from text -> number.
#    .cast("double") means "treat this as a number that can have decimals"
df_derm = df_derm.withColumn("price", F.col("price").cast("double"))

# 3) Confirm it worked: the type should now say 'double' instead of 'string'
df_derm.select("price").printSchema()

price
129.0
41.0
135.0
102.0
90.0
188.0
69.0
115.0
154.0
88.0


root
 |-- price: double (nullable = true)



In [0]:
df_derm.printSchema()

root
 |-- availability: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- category: string (nullable = true)
 |-- condition: string (nullable = true)
 |-- currency: string (nullable = true)
 |-- description: string (nullable = true)
 |-- how_to_use: string (nullable = true)
 |-- images: string (nullable = true)
 |-- ingredients: string (nullable = true)
 |-- price: double (nullable = true)
 |-- product_id: string (nullable = true)
 |-- range: string (nullable = true)
 |-- rating_value: double (nullable = true)
 |-- raw_how_to_use: string (nullable = true)
 |-- raw_ingredients: string (nullable = true)
 |-- raw_skin_type_and_concerns: string (nullable = true)
 |-- review_count: long (nullable = true)
 |-- scraped_at: string (nullable = true)
 |-- skin_type_and_concerns: string (nullable = true)
 |-- sku: string (nullable = true)
 |-- title: string (nullable = true)
 |-- uniq_id: string (nullable = true)
 |-- url: string (nullable = true)
 |-- volume: string (nullable = 

In [ ]:
df_derm.filter(F.col("description") != F.trim(F.col("description"))).select("description").display()

In [0]:

### Dermstore cleaning — Step 3: Split ingredients into a list

# Why: right now "ingredients" is ONE long blob of text, so the
# computer can't look inside it to find a specific active like
# "Niacinamide". Splitting turns the blob into a LIST of separate
# ingredients, which is what the recommender needs to search.
#
# The catch (from profiling): these lists mix TWO separators —
# commas AND line breaks. So we split on either one.

# Split "ingredients" into a new column "ingredients_list".
# Pattern  r"\s*[,\n]\s*"  means:
#   [,\n]  -> cut on a comma OR a line break
#   \s*    -> also remove any spaces around the cut
# We write to a NEW column so the original text stays intact (safer).
df_derm = df_derm.withColumn(
    "ingredients_list",
    F.split(F.col("ingredients"), r"\s*[,\n]\s*")
)

# Check it worked: ingredients should now show as lists [ ... ]
# with each ingredient as a separate item.
df_derm.select("ingredients_list").show(5, truncate=80)

+--------------------------------------------------------------------------------+
|                                                                ingredients_list|
+--------------------------------------------------------------------------------+
|[Aqua (Water), Acetyl Glucosamine, Cyclopentasiloxane, Triethyl Citrate, Cete...|
|[Silicone AA Battery., For the latest information, it is recommended to revie...|
|[Caprylic/Capric Triglyceride, Eucalyptus Globulus Leaf Oil, Juniperus Mexica...|
|[Water/Aqua/Eau, Lauramidopropyl Betaine*, Sodium Cocoyl Isethionate*, Glycer...|
|[Super Rich Repair: Water/Aqua/Eau, Isohexadecane, Dipropylene Glycol, Capryl...|
+--------------------------------------------------------------------------------+
only showing top 5 rows


In [0]:
# ============================================================
# Dermstore cleaning — Step 4: Drop products with no ingredients
# ============================================================
# Why: if a product has no ingredients, the recommender has nothing
# to match on — it's useless for our purpose. So we remove those rows.

# Count how many we have BEFORE dropping
print("Rows before:", df_derm.count())

# Keep only rows where ingredients is NOT null AND not an empty string.
#   F.col("ingredients").isNotNull()  -> has a value
#   F.trim(...) != ""                 -> isn't just blank/empty
df_derm = df_derm.filter(
    F.col("ingredients").isNotNull() & (F.trim(F.col("ingredients")) != "")
)

# Count AFTER dropping — the difference is how many we removed
print("Rows after: ", df_derm.count())

Rows before: 126
Rows after:  120


In [0]:
# Check what columns might tell us the product TYPE
df_derm.printSchema()

root
 |-- availability: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- category: string (nullable = true)
 |-- condition: string (nullable = true)
 |-- currency: string (nullable = true)
 |-- description: string (nullable = true)
 |-- how_to_use: string (nullable = true)
 |-- images: string (nullable = true)
 |-- ingredients: string (nullable = true)
 |-- price: double (nullable = true)
 |-- product_id: string (nullable = true)
 |-- range: string (nullable = true)
 |-- rating_value: double (nullable = true)
 |-- raw_how_to_use: string (nullable = true)
 |-- raw_ingredients: string (nullable = true)
 |-- raw_skin_type_and_concerns: string (nullable = true)
 |-- review_count: long (nullable = true)
 |-- scraped_at: string (nullable = true)
 |-- skin_type_and_concerns: string (nullable = true)
 |-- sku: string (nullable = true)
 |-- title: string (nullable = true)
 |-- uniq_id: string (nullable = true)
 |-- url: string (nullable = true)
 |-- volume: string (nullable = 

In [0]:
# Save cleaned Dermstore as a permanent "silver" Delta table
df_derm.write.mode("overwrite").saveAsTable("dermstore.dermstoreschema.silver_dermstore")
print("Saved! silver_dermstore is now permanent.")

Saved! silver_dermstore is now permanent.


### **Sephora**

In [ ]:
df_seph = spark.read.csv("/Volumes/dermstore/dermstoreschema/dermstorevolume/myfiles/sephora/product_info.csv")
display (df_seph)

# See the columns so we know what we're working with
df_seph.printSchema()

In [ ]:
df_seph=spark.read.\
option("header","true")\
.csv("/Volumes/dermstore/dermstoreschema/dermstorevolume/myfiles/sephora/product_info.csv")
display(df_seph)

In [0]:
df_seph.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- brand_id: string (nullable = true)
 |-- brand_name: string (nullable = true)
 |-- loves_count: string (nullable = true)
 |-- rating: string (nullable = true)
 |-- reviews: string (nullable = true)
 |-- size: string (nullable = true)
 |-- variation_type: string (nullable = true)
 |-- variation_value: string (nullable = true)
 |-- variation_desc: string (nullable = true)
 |-- ingredients: string (nullable = true)
 |-- price_usd: string (nullable = true)
 |-- value_price_usd: string (nullable = true)
 |-- sale_price_usd: string (nullable = true)
 |-- limited_edition: string (nullable = true)
 |-- new: string (nullable = true)
 |-- online_only: string (nullable = true)
 |-- out_of_stock: string (nullable = true)
 |-- sephora_exclusive: string (nullable = true)
 |-- highlights: string (nullable = true)
 |-- primary_category: string (nullable = true)
 |-- secondary_category: string (nullable = tru

In [0]:
# ============================================================
# SEPHORA cleaning — Step 2: Fix brand casing
# ============================================================
# Same idea as Dermstore, but Sephora's brand column is called
# "brand_name" (not "brand"). initcap standardizes the casing so
# "NEOSTRATA" and "neostrata" both become "Neostrata".

# Count distinct brands before
print("Distinct brands BEFORE:", df_seph.select("brand_name").distinct().count())

# Trim spaces, then standardize casing
df_seph = df_seph.withColumn("brand_name", F.initcap(F.trim(F.col("brand_name"))))

# Count after — if it drops, we merged casing duplicates
print("Distinct brands AFTER: ", df_seph.select("brand_name").distinct().count())

Distinct brands BEFORE: 305
Distinct brands AFTER:  305


In [0]:
# ============================================================
# SEPHORA cleaning — Step 3: Convert price to a number
# ============================================================
# Sephora's price column is "price_usd". First peek at the values
# to confirm they're clean numbers (no "$" signs or junk).
df_seph.select("price_usd").distinct().show(20, truncate=False)

+---------+
|price_usd|
+---------+
|210      |
|56       |
|120      |
|155      |
|110      |
|149      |
|245      |
|19.95    |
|35       |
|55       |
|14       |
|95       |
|105      |
|62       |
|16       |
|64       |
|52       |
|49.95    |
|125      |
|12       |
+---------+
only showing top 20 rows


In [0]:
# ============================================================
# SEPHORA cleaning — Step 3 (cont.): Convert price_usd to a number
# ============================================================
# Prices are clean plain numbers, so we just cast text -> double.
df_seph = df_seph.withColumn("price_usd", F.col("price_usd").cast("double"))

# Confirm it's now a number (should say price_usd: double)
df_seph.select("price_usd").printSchema()

root
 |-- price_usd: double (nullable = true)



In [0]:
# ============================================================
# SEPHORA cleaning — Step 3 (cont.): Convert price_usd to a number
# ============================================================
# Prices are clean plain numbers, so we just cast text -> double.
df_seph = df_seph.withColumn("price_usd", F.col("price_usd").cast("double"))

# Confirm it's now a number (should say price_usd: double)
df_seph.select("price_usd").printSchema()

root
 |-- price_usd: double (nullable = true)



In [0]:
# ============================================================
# SEPHORA cleaning — Step 4a: Look at the ingredient format first
# ============================================================
# Before splitting, see how Sephora separates ingredients.
# (Dermstore mixed commas and line breaks — Sephora might differ.)
df_seph.select("ingredients").show(5, truncate=120)

+------------------------------------------------------------------------------------------------------------------------+
|                                                                                                             ingredients|
+------------------------------------------------------------------------------------------------------------------------+
|['Capri Eau de Parfum:', 'Alcohol Denat. (SD Alcohol 39C), Parfum (Fragrance) D-Limonene, Linalool, Benzyl Salicylate...|
|['Alcohol Denat. (SD Alcohol 39C), Parfum (Fragrance) Ethylhexyl Methoxycinnamate, Ethylhexyl Salicylate, Butyl Metho...|
|['Alcohol Denat. (SD Alcohol 39C), Parfum (Fragrance) D-Limonene, Ethylhexyl Methoxycinnamate, Butyl Methoxydibenzoyl...|
|['Alcohol Denat. (SD Alcohol 39C), Parfum (Fragrance) Coumarin, Ethylhexyl Methoxycinnamate, Butyl Methoxydibenzoylme...|
|['Alcohol Denat. (SD Alcohol 39C), Parfum (Fragrance) Octoxynol-10, D-Limonene, Ethylhexyl Methoxycinnamate, Butyl Me...|
+---------------

In [0]:
# See ONE full ingredients value, nothing cut off, so we understand
# the exact format (brackets, quotes, separators).
df_seph.select("ingredients").filter(F.col("ingredients").isNotNull()).first()[0]

"['Capri Eau de Parfum:', 'Alcohol Denat. (SD Alcohol 39C), Parfum (Fragrance) D-Limonene, Linalool, Benzyl Salicylate, Ethylhexyl Methoxycinnamate, Butyl Methoxydibenzoylmethane, Ethylhexyl Salicylate, Benzl Benzoate, Citral, Geraniol, Eugenol, Benzyl Alcohol, Farnesol, Citronellol, Isoeugenol.', 'Invisible Post Eau de Parfum:', 'Alcohol Denat. (SD Alcohol 39C), Parfum (Fragrance) Ethylhexyl Methoxycinnamate, Ethylhexyl Salicylate, Butyl Methoxydibenzoylmethane, Benzyl Benzoate, Citral, Coumarin, Limonene, Alpha-Isomethyl Ionone, Linalool.', 'Kashbah Eau de Parfum:', 'Alcohol Denat. (SD Alcohol 39C), Parfum (Fragrance) Coumarin, Ethylhexyl Methoxycinnamate, Butyl Methoxydibenzoylmethane, Ethylhexyl Salicylate, D-Limonene, Eugenol, Linalool, Citronellol, Geraniol, Cinnamal, Citral.', 'L’Air Barbes Eau de Parfum:', 'Alcohol Denat. (SD Alcohol 39C), Parfum (Fragrance) Benzyl Salicylate, D-Limonene, Ethylhexyl Methoxycinnamate, Butyl Methoxydibenzoylmethane, Ethylhexyl Salicylate, Linaloo

In [0]:
# ============================================================
# SEPHORA cleaning — Step 4b: Clean & split ingredients
# ============================================================
# Sephora stores ingredients as a LIST-LOOKING STRING:
#   "['Ingredient A, Ingredient B', 'more stuff']"
# with leftover brackets [ ] and quotes ' '. We:
#   1) strip the wrapping brackets and quotes
#   2) split on commas into a real list
#
# Note: some fragrance "set" products bundle multiple sub-lists and
# won't parse perfectly — that's OK, those get filtered out later
# by ingredient validation (they have no skincare actives).

# 1) Remove the [ ] brackets and the ' quote characters.
#    regexp_replace finds characters and replaces them with nothing ("").
#    The pattern [\[\]'] means "any of: [ , ] , ' "
df_seph = df_seph.withColumn(
    "ingredients_clean",
    F.regexp_replace(F.col("ingredients"), r"[\[\]']", "")
)

# 2) Now split the cleaned text on commas into a list.
#    \s*,\s*  means "a comma with optional spaces around it"
df_seph = df_seph.withColumn(
    "ingredients_list",
    F.split(F.col("ingredients_clean"), r"\s*,\s*")
)

# Check the result — should be clean lists without brackets/quotes
df_seph.select("ingredients_list").show(3, truncate=100)

+----------------------------------------------------------------------------------------------------+
|                                                                                    ingredients_list|
+----------------------------------------------------------------------------------------------------+
|[Capri Eau de Parfum:, Alcohol Denat. (SD Alcohol 39C), Parfum (Fragrance) D-Limonene, Linalool, ...|
|[Alcohol Denat. (SD Alcohol 39C), Parfum (Fragrance) Ethylhexyl Methoxycinnamate, Ethylhexyl Sali...|
|[Alcohol Denat. (SD Alcohol 39C), Parfum (Fragrance) D-Limonene, Ethylhexyl Methoxycinnamate, But...|
+----------------------------------------------------------------------------------------------------+
only showing top 3 rows


In [0]:
# ============================================================
# SEPHORA cleaning — Step 5: Keep only the columns we need
# ============================================================
# Sephora has 27 columns, most of which our recommender doesn't use
# (loves_count, reviews, limited_edition, child_count, etc.).
# We keep just the useful ones and drop the rest.

df_seph = df_seph.select(
    "product_id",
    "product_name",
    "brand_name",
    "price_usd",
    "ingredients_list",        # our cleaned ingredient list
    "primary_category",        # product-type categories (skincare vs not)
    "secondary_category",
    "tertiary_category"
)

# Confirm: should now show just these 8 columns
df_seph.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- brand_name: string (nullable = true)
 |-- price_usd: double (nullable = true)
 |-- ingredients_list: array (nullable = true)
 |    |-- element: string (containsNull = false)
 |-- primary_category: string (nullable = true)
 |-- secondary_category: string (nullable = true)
 |-- tertiary_category: string (nullable = true)



In [ ]:
display(df_seph)

In [0]:
# ============================================================
# SEPHORA cleaning — Step 7: Save as a Delta table (silver layer)
# ============================================================
# Save the cleaned Sephora data permanently as "silver_sephora",
# our second silver table (matching silver_dermstore).
df_seph.write.mode("overwrite").saveAsTable("dermstore.dermstoreschema.silver_sephora")

print("Saved! silver_sephora is now permanent.")

Saved! silver_sephora is now permanent.


### Cosing

In [0]:
BASE = "/Volumes/dermstore/dermstoreschema/dermstorevolume/myfiles"

cosing_path = f"{BASE}/cosing/COSING_Ingredients-Fragrance Inventory_v2.csv"

df_cos = spark.read.option("header", True).csv(cosing_path)

print("Row count:", df_cos.count())
print("Column count:", len(df_cos.columns))
df_cos.printSchema()

Row count: 31759
Column count: 10
root
 |-- COSING Ref No: string (nullable = true)
 |-- INCI name: string (nullable = true)
 |-- INN name: string (nullable = true)
 |-- Ph. Eur. Name: string (nullable = true)
 |-- CAS No: string (nullable = true)
 |-- EC No: string (nullable = true)
 |-- Chem/IUPAC Name / Description: string (nullable = true)
 |-- Restriction: string (nullable = true)
 |-- Function: string (nullable = true)
 |-- Update Date: string (nullable = true)



In [ ]:
display(df_cos)

In [0]:
# ============================================================
# COSING cleaning — Step 2: Trim whitespace + keep needed columns
# ============================================================
# Profiling found INCI names have leading spaces (e.g. "  NIACINAMIDE").
# Untrimmed, lookups would fail. We trim, and also uppercase the name
# so matching later is consistent. Note: column names with spaces must
# be wrapped in backticks `like this`.

df_cos = df_cos.select(
    F.upper(F.trim(F.col("`INCI name`"))).alias("inci_name"),   # cleaned + uppercased, renamed
    F.trim(F.col("`Function`")).alias("function"),              # cleaned, renamed
    F.trim(F.col("`Restriction`")).alias("restriction")         # cleaned, renamed
)

# Check the result — names should be clean (no leading spaces), uppercase
df_cos.show(10, truncate=60)

+------------------------------------------------------------+------------------------------------------------+-----------+
|                                                   inci_name|                                        function|restriction|
+------------------------------------------------------------+------------------------------------------------+-----------+
|     DISODIUM TETRAMETHYLHEXADECENYLCYSTEINE FORMYLPROLINATE|                                 SKIN PROTECTING|       NULL|
|(ANGELICA ACUTILOBA/PAEONIA LACTIFLORA) ROOT/CNIDIUM OFFI...|                   SKIN CONDITIONING - EMOLLIENT|       NULL|
|(ARCTIUM LAPPA/RAPHANUS SATIVUS/MALVA VERTICILLATA) SEED OIL|                   SKIN CONDITIONING - EMOLLIENT|       NULL|
|(LIQUIDAMBAR STYRACIFLUA/TRIBULUS TERRESTRIS) FRUIT/PAEON...|                               SKIN CONDITIONING|       NULL|
|                   ACRYLATES/VA/VINYL NEODECANOATE COPOLYMER|                                     PLASTICISER|       NULL|
|       

In [0]:
# ============================================================
# COSING cleaning — Step 3: Save as a Delta table (silver layer)
# ============================================================
# Save the cleaned ingredient reference permanently as "silver_cosing",
# our third and final silver table.
df_cos.write.mode("overwrite").saveAsTable("dermstore.dermstoreschema.silver_cosing")

print("Saved! silver_cosing is now permanent.")

Saved! silver_cosing is now permanent.
